# Text-to-SQL Evaluation — Football Dataset

**Approach:** LangGraph-based multi-step reasoning  
**Dataset:** Football Database (`exp_v3` schema, PostgreSQL)  

## Pipeline overview

```
Question
   │
   ▼
select_tables  ─► identify which tables are relevant
   │
   ▼
generate_sql   ─► produce a PostgreSQL SELECT statement
   │
   ▼
execute_sql    ─► run query against the DB
   │
   ├─(error, retry < 2)─► fix_sql ─► execute_sql  (retry loop)
   │
   └─(success / max retries)─► generate_answer
                                    │
                                    ▼
                               Natural-language answer
```

In [1]:
# Install required packages
%pip install -qU "sqlalchemy>=2" psycopg2-binary \
    langchain-core langchain-community langchain-ollama langchain-openai \
    langgraph python-dotenv pandas


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Database Connection

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase

load_dotenv()

username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host     = os.getenv("DB_HOST")
port     = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

database_uri = f"postgresql://{username}:{password}@{host}:{port}/{database}"

try:
    sqldb = SQLDatabase.from_uri(database_uri, schema="exp_v3")
    print("Successfully connected to the Football Database!")
    names = sqldb.get_usable_table_names()
    print(f"Usable Tables ({len(names)}): {names}")
except Exception as e:
    print(f"Connection failed. Error: {e}")

Successfully connected to the Football Database!
Usable Tables (15): ['club', 'club_league_history', 'coach', 'coach_club_team', 'league', 'match_fact', 'national_opponent_team', 'national_team', 'player', 'player_club_team', 'player_fact', 'plays_match', 'stadium', 'world_cup', 'world_cup_result']


In [3]:
schema_snippet = sqldb.get_table_info()
print("\n--- Schema Snippet (first 2000 chars) ---")
print(schema_snippet[:2000])


--- Schema Snippet (first 2000 chars) ---

CREATE TABLE exp_v3.club (
	club_id VARCHAR NOT NULL, 
	club_name VARCHAR NOT NULL, 
	country VARCHAR, 
	found_year INTEGER, 
	CONSTRAINT club_pk PRIMARY KEY (club_id)
)

/*
3 rows from club table:
club_id	club_name	country	found_year
Q18741	Tottenham Hotspur F.C.	United Kingdom	1882
Q18708	Fulham F.C.	United Kingdom	1879
Q301973	Futbol Club Andorra Veterans	Andorra	1996
*/


CREATE TABLE exp_v3.club_league_history (
	club_id VARCHAR, 
	league_id VARCHAR, 
	start_year INTEGER, 
	end_year INTEGER, 
	CONSTRAINT club_league_history_club_id_fk FOREIGN KEY(club_id) REFERENCES exp_v3.club (club_id), 
	CONSTRAINT club_league_history_league_id_fk FOREIGN KEY(league_id) REFERENCES exp_v3.league (league_id)
)

/*
3 rows from club_league_history table:
club_id	league_id	start_year	end_year
Q1124841	Q12837728	2001	0
Q153539	Q82595	2021	2022
Q56734527	Q237181	2021	2022
*/


CREATE TABLE exp_v3.coach (
	nickname VARCHAR, 
	coach_id INTEGER NOT NULL, 
	coun

## 2. LLM Configuration

Set `LLM_PROVIDER=openai` in your `.env` to use OpenAI (requires `OPENAI_API_KEY`).  
Default: Ollama with `llama3.2:1b` (must be running locally on port 11434).
Override the model name via `OLLAMA_MODEL` or `OPENAI_MODEL`.

In [32]:
from dotenv import load_dotenv
import os

load_dotenv()

LLM_PROVIDER  = os.getenv("LLM_PROVIDER", "ollama")
OLLAMA_MODEL  = os.getenv("OLLAMA_MODEL", "llama3.2:1b")
OPENAI_MODEL  = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    print(f"Using OpenAI: {OPENAI_MODEL}")
else:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print(f"Using Ollama: {OLLAMA_MODEL}")

Using Ollama: llama3.2:1b


## 3. LangGraph Pipeline

### In-Context Learning (Few-Shot Prompting)

Three example question→SQL pairs are injected into the `generate_sql` prompt.
The examples are drawn from `data/dev.json` but **do not overlap** with the TEST_SET.
They cover easy (simple aggregation), medium (ILIKE lookup), and hard (3-table JOIN) patterns.


In [33]:
from typing import TypedDict, Optional, List


class TextToSQLState(TypedDict):
    question:        str
    relevant_tables: List[str]
    schema:          str
    sql_query:       str
    sql_result:      Optional[str]
    error:           Optional[str]
    retry_count:     int
    answer:          str

In [34]:
DB_SCHEMA  = "exp_v3"
ALL_TABLES = sqldb.get_usable_table_names()

# ── Few-shot examples for in-context learning ────────────────────────────
# Drawn from dev.json; none overlap with TEST_SET to avoid data leakage.
FEW_SHOT_EXAMPLES = """
-- Example 1 (easy): simple aggregation
Question: How often did Italy participate in the World Cup?
SQL: SELECT count(*) FROM exp_v3.national_team WHERE teamname = 'Italy'

-- Example 2 (medium): case-insensitive lookup on a single table
Question: In which country is A.C. Milan playing?
SQL: SELECT country FROM exp_v3.club WHERE club_name ILIKE '%A.C. Milan%'

-- Example 3 (hard): three-table JOIN with filters
Question: Show the names of all players from France in 2018.
SQL: SELECT T1.player_name
     FROM exp_v3.player AS T1
     JOIN exp_v3.player_fact AS T3 ON T1.player_id = T3.player_id
     JOIN exp_v3.national_team AS T2 ON T3.team_id = T2.team_id
     WHERE T2.teamname = 'France' AND T2.year = 2018
"""


def _clean_sql(text: str) -> str:
    """Strip markdown code fences that LLMs sometimes add."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        end = -1 if lines[-1].strip() == "```" else len(lines)
        text = "\n".join(lines[1:end])
    return text.strip()


# ── Node 1: select_tables ────────────────────────────────────────────────
def select_tables(state: TextToSQLState) -> dict:
    prompt = (
        "You are a SQL expert helping select relevant database tables.\n\n"
        f"Available tables: {', '.join(ALL_TABLES)}\n\n"
        f"Question: {state['question']}\n\n"
        "List ONLY the table names needed to answer this question, "
        "separated by commas. No explanation — just the table names."
    )
    response = llm.invoke(prompt)
    selected = [t.strip() for t in response.content.split(",")]
    valid    = [t for t in selected if t in ALL_TABLES] or ALL_TABLES
    schema   = sqldb.get_table_info(valid)
    return {"relevant_tables": valid, "schema": schema}


# ── Node 2: generate_sql (with few-shot examples) ────────────────────────
def generate_sql(state: TextToSQLState) -> dict:
    prompt = (
        "You are a PostgreSQL expert. Generate a valid SQL SELECT query "
        "to answer the question below.\n\n"
        f"Schema (tables in schema '{DB_SCHEMA}'):\n{state['schema']}\n\n"
        "Rules:\n"
        f"- Always prefix table names with the schema: {DB_SCHEMA}.table_name\n"
        "- Return ONLY the raw SQL — no markdown, no code fences, no explanation\n"
        "- The query must be a SELECT statement\n"
        "- Use standard PostgreSQL syntax\n\n"
        f"Here are some examples:\n{FEW_SHOT_EXAMPLES}\n"
        f"Now answer this question:\n"
        f"Question: {state['question']}\n\nSQL:"
    )
    response = llm.invoke(prompt)
    sql = _clean_sql(response.content)
    return {"sql_query": sql, "error": None}


# ── Node 3: execute_sql ──────────────────────────────────────────────────
def execute_sql(state: TextToSQLState) -> dict:
    try:
        result = sqldb.run(state["sql_query"])
        return {"sql_result": str(result), "error": None}
    except Exception as exc:
        return {
            "sql_result": None,
            "error": str(exc),
            "retry_count": state.get("retry_count", 0) + 1,
        }


# ── Node 4: fix_sql ──────────────────────────────────────────────────────
def fix_sql(state: TextToSQLState) -> dict:
    prompt = (
        "You are a PostgreSQL expert. A SQL query failed — fix it.\n\n"
        f"Schema (tables in schema '{DB_SCHEMA}'):\n{state['schema']}\n\n"
        f"Original question: {state['question']}\n\n"
        f"Failing SQL:\n{state['sql_query']}\n\n"
        f"Error message:\n{state['error']}\n\n"
        "Rules:\n"
        f"- Always prefix table names with the schema: {DB_SCHEMA}.table_name\n"
        "- Return ONLY the corrected raw SQL — no markdown, no code fences\n\n"
        "Fixed SQL:"
    )
    response = llm.invoke(prompt)
    sql = _clean_sql(response.content)
    return {"sql_query": sql}


# ── Node 5: generate_answer ──────────────────────────────────────────────
def generate_answer(state: TextToSQLState) -> dict:
    if state.get("error") and not state.get("sql_result"):
        return {"answer": f"[Could not answer — SQL error]: {state['error']}"}
    prompt = (
        "Answer the following question concisely based on the SQL result.\n\n"
        f"Question: {state['question']}\n"
        f"SQL result: {state['sql_result']}\n\n"
        "Answer:"
    )
    response = llm.invoke(prompt)
    return {"answer": response.content.strip()}


print("All graph nodes defined (few-shot examples loaded).")

All graph nodes defined (few-shot examples loaded).


In [35]:
from langgraph.graph import StateGraph, START, END


def _route_after_execution(state: TextToSQLState) -> str:
    if state.get("error") and state.get("retry_count", 0) < 2:
        return "fix_sql"
    return "generate_answer"


builder = StateGraph(TextToSQLState)
builder.add_node("select_tables",  select_tables)
builder.add_node("generate_sql",   generate_sql)
builder.add_node("execute_sql",    execute_sql)
builder.add_node("fix_sql",        fix_sql)
builder.add_node("generate_answer", generate_answer)

builder.add_edge(START, "select_tables")
builder.add_edge("select_tables", "generate_sql")
builder.add_edge("generate_sql",  "execute_sql")
builder.add_conditional_edges(
    "execute_sql",
    _route_after_execution,
    {"fix_sql": "fix_sql", "generate_answer": "generate_answer"},
)
builder.add_edge("fix_sql",        "execute_sql")
builder.add_edge("generate_answer", END)

graph = builder.compile()
print("Graph compiled successfully.\n")
print(graph.get_graph().draw_mermaid())

Graph compiled successfully.

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	select_tables(select_tables)
	generate_sql(generate_sql)
	execute_sql(execute_sql)
	fix_sql(fix_sql)
	generate_answer(generate_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> select_tables;
	execute_sql -.-> fix_sql;
	execute_sql -.-> generate_answer;
	fix_sql --> execute_sql;
	generate_sql --> execute_sql;
	select_tables --> generate_sql;
	generate_answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4. Demo — Single Query

In [36]:
def run_query(question: str, verbose: bool = True) -> dict:
    """Run a natural-language question through the Text-to-SQL pipeline."""
    initial: TextToSQLState = {
        "question":        question,
        "relevant_tables": [],
        "schema":          "",
        "sql_query":       "",
        "sql_result":      None,
        "error":           None,
        "retry_count":     0,
        "answer":          "",
    }
    result = graph.invoke(initial)
    if verbose:
        print(f"Question : {result['question']}")
        print(f"Tables   : {result['relevant_tables']}")
        print(f"SQL      :\n  {result['sql_query']}")
        print(f"DB result: {result['sql_result']}")
        if result.get('error'):
            print(f"Error    : {result['error']}")
        print(f"Answer   : {result['answer']}")
    return result


# Demo
_ = run_query("Which club was founded first and in which year?")

Question : Which club was founded first and in which year?
Tables   : ['club_league_history', 'coach', 'club', 'league', 'national_opponent_team', 'player', 'plays_match', 'stadium', 'world_cup']
SQL      :
  SELECT club_name FROM exp_v3.club ORDER BY found_year LIMIT 1
DB result: [('R.S. Ginnastica Torino',)]
Answer   : The club that was founded first is R.S. Ginnastica Torino, and it was founded in 1909.


## 5. Test-Set Evaluation

**Metric: Execution Accuracy**  
A predicted query is counted as *correct* when its executed result set matches the gold result set exactly.  

**Test set:** 18 questions drawn from `data/dev.json` (Football DB, `exp_v3` schema).  
Split: 9 easy/medium + 9 hard/extra-hard (50 % / 50 %).


In [37]:
# 18 questions from data/dev.json
# Split: 9 easy/medium  |  9 hard/extra-hard
TEST_SET = [
    # ── EASY (3) ──────────────────────────────────────────────────────────
    {
        "id": 779, "hardness": "easy",
        "question": "Where was the world cup in 2018?",
        "gold_sql":  "SELECT venue FROM world_cup WHERE year = 2018",
    },
    {
        "id": 268, "hardness": "easy",
        "question": "Show Brazilian club names",
        "gold_sql":  "SELECT club_name FROM club WHERE country = 'Brazil'",
    },
    {
        "id": 297, "hardness": "easy",
        "question": "Show number of teams in 2022",
        "gold_sql":  "SELECT count(teamname) FROM national_team WHERE year = '2022'",
    },
    # ── MEDIUM (6) ────────────────────────────────────────────────────────
    {
        "id": 89, "hardness": "medium",
        "question": "Give me the year of the first world cup played.",
        "gold_sql":  "SELECT year FROM world_cup ORDER BY year ASC LIMIT 1",
    },
    {
        "id": 506, "hardness": "medium",
        "question": "Which national team won the world cup in 1998",
        "gold_sql":  (
            "SELECT T1.teamname FROM national_team AS T1 "
            "JOIN world_cup_result AS T2 ON T1.team_id = T2.team_id "
            "WHERE T2.year = 1998 AND T2.winner = 'true'"
        ),
    },
    {
        "id": 708, "hardness": "medium",
        "question": "Who won the 2014 world cup?",
        "gold_sql":  (
            "SELECT T1.teamname FROM national_team AS T1 "
            "JOIN world_cup_result AS T2 ON T1.team_id = T2.team_id "
            "WHERE T2.year = 2014 AND T2.winner = 'true'"
        ),
    },
    {
        "id": 569, "hardness": "medium",
        "question": "Which team is in Group A in 2018",
        "gold_sql":  (
            "SELECT DISTINCT T2.teamname FROM plays_match AS T1 "
            "JOIN national_team AS T2 ON T1.team_id = T2.team_id "
            "WHERE T1.year = 2018 AND T1.stage = 'Group A'"
        ),
    },
    {
        "id": 861, "hardness": "medium",
        "question": "Which is the oldest club?",
        "gold_sql":  (
            "SELECT club_name, found_year FROM club "
            "ORDER BY found_year ASC LIMIT 1"
        ),
    },
    {
        "id": 605, "hardness": "medium",
        "question": "Which teams played semifinals in 2018?",
        "gold_sql":  (
            "SELECT DISTINCT T2.teamname FROM plays_match AS T1 "
            "JOIN national_team AS T2 ON T1.team_id = T2.team_id "
            "WHERE T1.year = 2018 AND T1.stage = 'Semi-finals'"
        ),
    },
    # ── HARD (5) ──────────────────────────────────────────────────────────
    {
        "id": 94, "hardness": "hard",
        "question": "How many games did Brazil win?",
        "gold_sql":  (
            "SELECT count(*) FROM national_team AS T1 "
            "JOIN plays_match AS T2 ON T1.team_id = T2.team_id "
            "JOIN national_opponent_team AS T3 ON T2.opponent_team_id = T3.team_id "
            "WHERE T1.teamname = 'Brazil' AND T2.team_goals > T2.opponent_team_goals"
        ),
    },
    {
        "id": 155, "hardness": "hard",
        "question": "How many times did England win the world cup",
        "gold_sql":  (
            "SELECT count(*) FROM world_cup_result AS T1 "
            "JOIN national_team AS T2 ON T1.team_id = T2.team_id "
            "WHERE T2.teamname = 'England' AND T1.winner = 'True'"
        ),
    },
    {
        "id": 259, "hardness": "hard",
        "question": "Show all national teams who have won more than 2 world cups",
        "gold_sql":  (
            "SELECT T2.teamname FROM world_cup_result AS T1 "
            "JOIN national_team AS T2 ON T1.team_id = T2.team_id "
            "WHERE T1.winner = 'true' GROUP BY T2.teamname "
            "HAVING COUNT(T1.year) > 2"
        ),
    },
    {
        "id": 426, "hardness": "hard",
        "question": "What team shot the most goals in 2018",
        "gold_sql":  (
            "SELECT teamname, goals FROM national_team "
            "WHERE year = 2018 ORDER BY goals DESC LIMIT 1"
        ),
    },
    {
        "id": 534, "hardness": "hard",
        "question": "Which players played for Switzerland in 2014? Return the names of the players",
        "gold_sql":  (
            "SELECT DISTINCT p.player_name FROM player AS p "
            "JOIN player_fact AS pf ON p.player_id = pf.player_id "
            "JOIN national_team AS nt ON pf.team_id = nt.team_id "
            "WHERE nt.teamname = 'Switzerland' AND nt.year = 2014"
        ),
    },
    # ── EXTRA-HARD (4) ────────────────────────────────────────────────────
    {
        "id": 120, "hardness": "extra",
        "question": "How many goals did Ronaldo Score?",
        "gold_sql":  (
            "SELECT T2.player_name, count(*) FROM match_fact AS T1 "
            "JOIN player AS T2 ON T1.player_id = T2.player_id "
            "WHERE T1.goal = 'true' AND T2.player_name ILIKE '%Ronaldo%' "
            "GROUP BY T2.player_name"
        ),
    },
    {
        "id": 284, "hardness": "extra",
        "question": "Show me the name of the player with the most played matches.",
        "gold_sql":  (
            "SELECT t1.player_name, count(*) FROM player AS t1 "
            "JOIN match_fact AS t2 ON t1.player_id = t2.player_id "
            "GROUP BY t1.player_name ORDER BY count(*) DESC LIMIT 1"
        ),
    },
    {
        "id": 551, "hardness": "extra",
        "question": "Which stadium hosts the most games",
        "gold_sql":  (
            "SELECT T1.stadium_name, count(*) FROM stadium AS T1 "
            "JOIN plays_match AS T2 ON T1.stadium_id = T2.stadium_id "
            "GROUP BY T1.stadium_name ORDER BY count(*) DESC LIMIT 1"
        ),
    },
    {
        "id": 591, "hardness": "extra",
        "question": "Which Team won the most recent world cup and in which year did they win?",
        "gold_sql":  (
            "SELECT T1.teamname, T2.year FROM national_team AS T1 "
            "JOIN world_cup_result AS T2 ON T1.team_id = T2.team_id "
            "WHERE T2.winner = 'true' ORDER BY T2.year DESC LIMIT 1"
        ),
    },
]

print(f"Test set loaded: {len(TEST_SET)} questions")
by_hardness = {}
for q in TEST_SET:
    by_hardness.setdefault(q["hardness"], []).append(q["id"])
for h, ids in sorted(by_hardness.items()):
    print(f"  {h:8s}: {len(ids)} questions  (ids: {ids})")


Test set loaded: 18 questions
  easy    : 3 questions  (ids: [779, 268, 297])
  extra   : 4 questions  (ids: [120, 284, 551, 591])
  hard    : 5 questions  (ids: [94, 155, 259, 426, 534])
  medium  : 6 questions  (ids: [89, 506, 708, 569, 861, 605])


In [38]:
def _normalize(result) -> str:
    """Normalise a SQL result for comparison (lower-case, strip whitespace)."""
    return str(result).strip().lower().replace(" ", "")


def evaluate(test_set: list, verbose: bool = True) -> dict:
    """
    Evaluate execution accuracy over test_set.

    For each item:
      1. Execute the gold SQL to obtain the reference result.
      2. Run the pipeline to obtain a predicted SQL and result.
      3. Mark as correct if the normalised results match.

    Returns a dict with 'accuracy', 'correct', 'total', and 'details'.
    """
    correct = 0
    details = []

    for i, item in enumerate(test_set):
        question = item["question"]
        gold_sql = item["gold_sql"]

        if verbose:
            print(f"\n[{i + 1}/{len(test_set)}] {question}")

        # Gold result
        try:
            gold_result = sqldb.run(gold_sql)
        except Exception as exc:
            gold_result = f"GOLD_ERROR: {exc}"

        # Pipeline prediction
        pred        = run_query(question, verbose=False)
        pred_sql    = pred.get("sql_query", "")
        pred_result = pred.get("sql_result")
        pred_error  = pred.get("error")

        match = _normalize(gold_result) == _normalize(pred_result)
        if match:
            correct += 1

        details.append({
            "question":         question,
            "gold_sql":         gold_sql,
            "gold_result":      gold_result,
            "predicted_sql":    pred_sql,
            "predicted_result": pred_result,
            "error":            pred_error,
            "correct":          match,
        })

        if verbose:
            status = "CORRECT" if match else "WRONG"
            print(f"  Gold SQL      : {gold_sql}")
            print(f"  Predicted SQL : {pred_sql}")
            print(f"  Gold result   : {gold_result}")
            print(f"  Pred result   : {pred_result}")
            if pred_error:
                print(f"  Error         : {pred_error}")
            print(f"  Status        : {status}")

    accuracy = correct / len(test_set) if test_set else 0.0
    return {"accuracy": accuracy, "correct": correct, "total": len(test_set), "details": details}


print("Evaluation function ready.")

Evaluation function ready.


In [39]:
eval_results = evaluate(TEST_SET, verbose=True)

print(f"\n{'=' * 55}")
print(f"Execution Accuracy: {eval_results['correct']}/{eval_results['total']} "
      f"= {eval_results['accuracy']:.1%}")
print(f"{'=' * 55}")

  Gold SQL      : SELECT T1.teamname, T2.year FROM national_team AS T1 JOIN world_cup_result AS T2 ON T1.team_id = T2.team_id WHERE T2.winner = 'true' ORDER BY T2.year DESC LIMIT 1
  Predicted SQL : SELECT qualified_team FROM exp_v3.world_cup ORDER BY qualified_team DESC LIMIT 1
  Gold result   : [('Argentina', 2022)]
  Pred result   : [(32,)]
  Status        : WRONG

Execution Accuracy: 5/18 = 27.8%


In [40]:
import pandas as pd

df = pd.DataFrame([
    {
        "#":               i + 1,
        "Question":        r["question"],
        "Predicted SQL":   r["predicted_sql"],
        "Gold result":     r["gold_result"],
        "Predicted result":r["predicted_result"],
        "Correct":         "✓" if r["correct"] else "✗",
    }
    for i, r in enumerate(eval_results["details"])
])

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)
display(df)

n_correct = eval_results['correct']
n_total   = eval_results['total']
print(f"\nFinal Execution Accuracy: {n_correct}/{n_total} = {eval_results['accuracy']:.1%}")


Final Execution Accuracy: 5/18 = 27.8%
